# **DeepLIFT: Practice**

DeepLIFT to some extent continues the idea of Integrated gradients, focusing on summing activations relative to some "base" activations on a reference example. By the definition of the method in the [original paper](https://arxiv.org/abs/1704.02685), there are no restrictions on applying it to any data modality or model. But because of the specifics of the method, in its implementation it does not work for all architectures and, at the time of writing this course, it is adapted only to images. That is why this practice is also devoted to working with images :)


As before in the course, we will work with an implementation that helps to interpret pyTorch models. The already familiar [captum](https://captum.ai/) library will help us with this.

Deep LIFT is also available for use with models trained with the Keras framework. You can find that implementation in the library of the same name, [deeplift](https://github.com/kundajelab/deeplift).  

**An interesting and rare fact for XAI**: [DeepLIFT has been adapted](https://www.kaggle.com/code/marcellveiner/deeplift-exploration-on-genomic-data) for application to genomic data for Keras models and with the help of libraries for processing genomic data, for example [tangermeme](https://tangermeme.readthedocs.io/en/latest/index.html).

In this practice you will:
- Practise using the captum library
- Assess various baselines for the deepLIFT method

In [ ]:
!pip install captum -q

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime
import numpy as np

%matplotlib inline

import torch
from io import BytesIO
import requests
import torchvision
from torch.autograd import Variable
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

from torchvision.models import swin_t

from captum.attr import DeepLift
from captum.attr import visualization as viz

import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import alexnet

import urllib

url = "https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/imagenet_classes.txt"
urllib.request.urlretrieve(url, "imagenet_classes.txt")

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

This time we will work a bit more with the lovely piggy. We will do all the standard steps:
- loading the image
- writing the preprocessing functions
- preprocessing the image

In [ ]:
hog_url = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/hog.jpg'

hog_image_bytes = requests.get(hog_url).content
image = Image.open(BytesIO(hog_image_bytes))

plt.axis('off')
plt.imshow(image);

1. Complete the preprocess function by adding normalisation (``` transforms.Normalize(mean=[..., ..., ...], std=[..., ..., ...]) ```) of the image. Why do we know in advance which means and standard deviations to compute?

In [ ]:
#Preprocessing

preprocess = transforms.Compose([
   transforms.Resize((224,224)),
   transforms.ToTensor(),

# Your code here
)
])

display = transforms.Compose([transforms.Resize((224,224))])

In [ ]:
input = preprocess(image)

input.unsqueeze_(0);
input.requires_grad = True

In the previous practices we worked with models whose top-5 accuracy exceeds 90%. This time, for the sake of interest, we will look at a "weaker" model — AlexNet with a top-5 accuracy of 79%.

In [ ]:
alex_net = alexnet(weights='IMAGENET1K_V1')

# Inference mode is a must: in train mode BatchNorm uses the statistics of a batch
# of one image and Dropout keeps dropping units — the prediction and the attributions
# stop being reproducible.
alex_net.eval();

2. Get the model's prediction for the piggy. Which class does AlexNet predict?

In [ ]:
# Your code here
output =

print('Model prediction: ', torch.max(output, 1)[1])

3. Get and then decode, one by one, the top-5 predictions using the `categories` list.

In [ ]:
top_5_classes = torch.topk(output, k=5)

indexes, probabilities = # # Your code here

for i, j in zip(indexes,probabilities):
  print(i, categories[i], 'probability: ', j)

Let us look at what makes the model see a polecat in the piggy, using DeepLIFT.
*(an illustration for the walkthrough — run the cells below, the map is built right in the notebook)*

To begin with, as in the previous lesson, let us generate the baselines! This time we will add a new one — the per-channel mean computed on Imagenet.

In [ ]:
def gaussian_noise(x, var):
  """
  Gaussian noise based on the image x
  """

  return torch.normal(0, var, size=x.shape)


def random_baseline(x, low, high):
    """
    A random distribution based on the image x
    """

    return np.random.uniform(low, high, size=x.shape)

In [ ]:
zero_baseline = input * 0 #Zero baseline

noise_baseline = torch.ones_like(input) # Noise baseline
noise_baseline += gaussian_noise(input, 0.1)

mean_baseline = torch.ones_like(input) # Mean baseline
mean_baseline *= torch.mean(input, axis=1)

mean_imagenet_baseline = torch.ones_like(input) # Mean baseline IMAGENET
mean_imagenet_baseline[0, 0] = mean_imagenet_baseline[0, 0]*0.485
mean_imagenet_baseline[0, 1] = mean_imagenet_baseline[0, 1]*0.456
mean_imagenet_baseline[0, 2] =  mean_imagenet_baseline[0, 2]*0.406


random_baseline = random_baseline(x=input, low=0, high=15) # Random baseline
random_baseline = torch.from_numpy(random_baseline).float()

As in the practice on IG, let us make a function for convenient visualisation

In [ ]:
def vizualize_tensor(tensor):

  if len(tensor.shape) == 4:
    return tensor.squeeze(0).permute((1, 2, 0)).detach().numpy()
  else:
    return tensor.permute((1, 2, 0)).detach().numpy()

In [ ]:
# Creating the figure and a 1x5 grid
fig, axes = plt.subplots(1, 5, figsize=(16, 8))

# Displaying each image, row 1
axes[0].imshow(vizualize_tensor(zero_baseline))
axes[0].set_title('Zero baseline')

axes[1].imshow(vizualize_tensor(noise_baseline))
axes[1].set_title('Noise baseline')

axes[2].imshow(vizualize_tensor(mean_baseline))
axes[2].set_title('Mean baseline')

axes[3].imshow(vizualize_tensor(random_baseline))
axes[3].set_title('Random baseline');

axes[4].imshow(vizualize_tensor(mean_imagenet_baseline))
axes[4].set_title('Mean IMAGENET baseline');

As in the previous practice, to work with the gradient explanation method from captum you will need `DeepLIFT` (we have again already imported it in the first cell) and the .attribute method — with almost exactly the same parameters as for Integrated Gradients.



```
attribute(inputs, baselines=None, target=None, additional_forward_args=None, return_convergence_delta=False, custom_attribution_func=None)[source]¶

```
where:
- inputs — the input images
- baselines - the baselines of the images
- target — the class the explanation will be built for
- additional_forward_args — in case the model returns not only the standard output, but other Python objects as well
- return_convergence_delta — the difference between the total approximated and the true integrated gradients
- custom_attribution_func — in case you want to compute the attributions with a special method.

DeepLIFT itself also has hyperparameters:
- multiply_by_inputs (bool, optional) — indicates whether the input baseline should be taken into account. In the literature this is also known as local or global attribution ([here](https://arxiv.org/abs/1711.06104) you can read about it in more detail).  If the baseline input data is not taken into account, then this type of attribution method is also called local attribution, and if the opposite, then this type of attribution method is called global.
- eps (float, optional) — the value at which a change of the output/input data should be considered significant when computing the gradients for nonlinear layers. Useful for tuning deep models in order to avoid problems when computing the gradient. Default: 1e-10

In [ ]:
alex_net.zero_grad()
dl = DeepLift(alex_net) # Initialising DeepLIFT

Let us compute the attributions with deepLIFT. Here the code is no different from our practice on Integrated Gradients.

In [ ]:
dl_zero_attributions = dl.attribute(input, zero_baseline, 358)
dl_noise_attributions = dl.attribute(input, noise_baseline, 358)
dl_mean_attributions = dl.attribute(input, mean_baseline, 358)
dl_mean_imagenet_attributions = dl.attribute(input, mean_imagenet_baseline, 358)
dl_random_attributions = dl.attribute(input, random_baseline, 358)

**Visualisation. The built-in method.**

Now let us try to visualise the values directly.

In [ ]:
plt.imshow(vizualize_tensor(dl_zero_attributions));

A black square! But do all the values in it really indicate an absence of information?

Find the minimum and the maximum value in `dl_zero_attributions`. Give the minimum you found as the answer in the trainer.

In [ ]:
# Your code here

Probably, by this step you already guess what the problem with the black square is — the values are too small (the Warning, by the way, tells us about this), and in the previous practice we multiplied them by 10.

How do we capture the gradients? On the one hand, everything is already implemented in the `captum` library under the hood, and we can make use of that by using the `visualize_image_attr` method from the `viz` module in `captum`.

```
captum.attr.visualization.visualize_image_attr(attr,
                                              original_image=None,
                                              method='heat_map',
                                              sign='absolute_value',
                                              plt_fig_axis=None,
                                              outlier_perc=2, cmap=None,
                                              alpha_overlay=0.5,
                                              show_colorbar=False, title=None,
                                              fig_size=(6, 6), use_pyplot=True)
```

The method has many parameters and many of them are intuitively clear. In practice, what turns out to be very useful is not so much the use of the overlaid heatmap, as *the use of a mask* built from the resulting map on the original image. You can set the visualisation to be a mask (and in general the type of the visualisation) with the `method` hyperparameter.

In [ ]:
#Let us visualise all the maps with masks

_ = viz.visualize_image_attr(vizualize_tensor(dl_zero_attributions),
                             np.array(display(image)),
                             method="masked_image",
                             sign="absolute_value",
                             title="Masked DeepLift with zero baseline")

_ = viz.visualize_image_attr(vizualize_tensor(dl_noise_attributions),
                             np.array(display(image)),
                             method="masked_image",
                             sign="absolute_value",
                              title="Masked DeepLift with noise baseline")

_ = viz.visualize_image_attr(vizualize_tensor(dl_mean_attributions),
                             np.array(display(image)),
                             method="masked_image",
                             sign="absolute_value",
                             title="Masked DeepLift with mean baseline")

_ = viz.visualize_image_attr(vizualize_tensor(dl_mean_imagenet_attributions),
                             np.array(display(image)),
                             method="masked_image",
                             sign="absolute_value",
                             title="Masked DeepLift with mean imagenet baseline")

_ = viz.visualize_image_attr(vizualize_tensor(dl_random_attributions),
                             np.array(display(image)),
                             method="masked_image",
                             sign="absolute_value",
                            title="Masked DeepLift with random baseline")

Which of the baselines shows the most compressed information (produces a sparser map as the output)? Choose the answer in the trainer.

On the masks you can see that the piggy's colouring can vaguely resemble a polecat. However, such a conclusion is more of a hypothesis, which needs to be tested by fixing the data and experimenting with other images.

**Visualisation. Manual correction.**

Let us come back to the problem of small values. The second way is to rescale the values to a larger scale, while preserving their meaning. This can be done with, for example, MinMax Scaling:

$$MinMax(X) = \frac{X-min(X)}{max(X)-min(X)}$$

What will the maximum of such a transformation be?

Implement MinMax scaling as a function and apply it to all the maps.

In [ ]:
def MinMaxScaling(X):

  min_val = X.min()
  max_val = X.max()

  return (X - min_val)/(max_val - min_val)

In [ ]:
for attr in [dl_zero_attributions, dl_noise_attributions, dl_random_attributions, dl_mean_attributions, dl_mean_imagenet_attributions]:

  normalized_map =  MinMaxScaling(attr)
  plt.imshow(vizualize_tensor(normalized_map))
  plt.show()

This is where our practice ends. Thank you! We would be glad if you tried experimenting and shared what you got when experimenting with your own images!

In [ ]:
plt.imshow(display(image))
plt.title('Thank you!');